In [1]:
year = 1993
month = 1

In [2]:
# Parameters
year = 1996
month = 9


In [3]:
import copernicusmarine
import xarray as xr
import matplotlib.pyplot as plt
from cmocean import cm 
import numpy as np
import pandas as pd

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

In [4]:
def prepare_ocean_dataset(ds):
    """
    Prepare ocean dataset with proper coordinates, masks, and vertical velocity calculation.
    
    Parameters
    ----------
    ds : xarray.Dataset
        Input dataset with dimensions (depth, latitude, longitude) and variables (uo, vo)
    
    Returns
    -------
    xarray.Dataset
        Processed dataset with renamed dimensions, calculated masks, and vertical velocity
    """
    ds_i = ds
    _lat = ds.latitude
    _lon = ds.longitude
    _zt = ds.depth
    
    ds_i = ds_i.rename({"depth": "k", "latitude":"j", "longitude":"i","uo":"uf", "vo":"vf"})
    ds_i = ds_i.assign_coords(
        k=np.arange(ds_i.sizes["k"]),
        j=np.arange(ds_i.sizes["j"]),
        i=np.arange(ds_i.sizes["i"]),
        depth_t=("k", _zt.data),
        latitude_f = ("j", _lat.data),
        longitude_f = ("i", _lon.data),
    )
    
    
    ## Calculate F and T mask
    ds_i = ds_i.assign(fmask = ds_i.uf.isel(time=0,drop=True).notnull())
    
    ds_i = ds_i.assign(
        tmask=(
            ds_i.fmask.shift(i=0,j=0)
            | ds_i.fmask.shift(i=-1,j=-1).fillna(False)
            | ds_i.fmask.shift(i=0, j=-1).fillna(False)
            | ds_i.fmask.shift(i=-1,j=-1).fillna(False)
        ).astype(bool)
    )
    
    ## Calculate U and V faces
    ds_i = ds_i.assign(
        u=(ds_i.uf.fillna(0) + ds_i.uf.shift(j=-1).fillna(0)) /2,
        v=(ds_i.vf.fillna(0) + ds_i.vf.shift(i=-1).fillna(0)) /2,
    )
    
    ## Calculate Zt
    zt = ds_i.depth_t.data
    zw = [zt[0]*2]
    
    
    for k in range(1,50):
        zw.append((zt[k] - zw[k-1])*2 + zw[k-1])
    
    ds_i = ds_i.assign_coords(depth_w = ("k",zw))
    
    ds_i = ds_i.assign_coords(
        longitude_u = ds_i.longitude_f,
        latitude_v =  ds_i.latitude_f,
        
        latitude_u = ds_i.latitude_f + 1/12/2, 
        longitude_v = ds_i.longitude_f + 1/12/2,
        
        latitude_t = ds_i.latitude_f + 1/12/2, 
        longitude_t = ds_i.longitude_f + 1/12/2,
    )
    
    R = 6371e3 
    
    ds_i = ds_i.assign_coords(
        dz_t = ds_i.depth_w - ds_i.depth_w.shift(k=1).fillna(0), 
        dx_t = np.deg2rad(1/12) * R * np.cos(np.deg2rad(ds_i.latitude_t)),
        dy_t = np.deg2rad(1/12) * R ,
        
    )
    
    ## we find the total volume flux - m3
    F_uv_vol = (
        ds_i.u * ds_i.dy_t * ds_i.dz_t - ds_i.u.shift(i=-1)* ds_i.dy_t * ds_i.dz_t 
        + ds_i.v * ds_i.dx_t * ds_i.dz_t - ds_i.v.shift(j=-1) * ds_i.dx_t * ds_i.dz_t
    ).fillna(0)
    
    #we divide the total flux by the volume (dx*dy*dz) - 1/s
    dw_by_dz = -F_uv_vol/ds_i.dx_t/ds_i.dy_t/ds_i.dz_t
    
    w = (dw_by_dz.fillna(0) * ds_i.dz_t.fillna(0)).cumsum('k').fillna(0).where(ds_i.tmask==1)
    
    #we get the tmask
    tmask = ds_i.tmask.compute()
    
    w_bottom=w.isel(k=tmask.sum('k')-1)
    w_correct = w - w_bottom / ds_i.dz_t.where(ds_i.tmask==1).sum('k') * ds_i.depth_w
    ds_i['w_c'] = w_correct
    
    ds_i = ds_i.drop_vars(['u','v','fmask','tmask'])

    # #1. We insert the 0m at z
    # k=np.arange(0,51,1)

    # #2. We linearly interpolate the U,V
    # ds_i_= ds_i.interp(k=np.arange(0,51,1))
    # ds_i_['w_c'][..., 0, :, :] = 0
    
    return ds_i

## Call CMEMS data

In [5]:
from datetime import datetime
import calendar

In [6]:
last_day = calendar.monthrange(year, month)[1]
start_date = f"{year}-{month:02d}-01T00:00:00"
end_date = f"{year}-{month:02d}-{last_day:02d}T23:59:59"

In [7]:
data_request = {
   "dataset_id_plume" : "cmems_mod_glo_phy_my_0.083deg_P1D-m",
   "dataset_version": "202311",
   "longitude" : [-100, -0], 
   "latitude" : [-50, 50],
   "time" : [start_date, end_date],
   "variables" : ["vo","uo"]
}

# Load xarray dataset
ds = copernicusmarine.open_dataset(
    dataset_id = data_request["dataset_id_plume"],
    minimum_longitude = data_request["longitude"][0],
    maximum_longitude = data_request["longitude"][1],
    minimum_latitude = data_request["latitude"][0],
    maximum_latitude = data_request["latitude"][1],
    start_datetime = data_request["time"][0],
    end_datetime = data_request["time"][1],
    variables = data_request["variables"],
    username = 'alizarbe',
    password = 'DoNuT_120197',
    chunk_size_limit = -1
)

# Print loaded dataset information
ds

INFO - 2025-09-09T00:51:15Z - Selected dataset version: "202311"


INFO - 2025-09-09T00:51:15Z - Selected dataset part: "default"


<xarray.Dataset> Size: 35GB
Dimensions:    (depth: 50, latitude: 1201, longitude: 1201, time: 30)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 5kB -50.0 -49.92 -49.83 ... 49.83 49.92 50.0
  * longitude  (longitude) float32 5kB -100.0 -99.92 -99.83 ... -0.08333 0.0
  * time       (time) datetime64[ns] 240B 1996-09-01 1996-09-02 ... 1996-09-30
Data variables:
    vo         (time, depth, latitude, longitude) float64 17GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
    uo         (time, depth, latitude, longitude) float64 17GB dask.array<chunksize=(2, 50, 512, 1201), meta=np.ndarray>
Attributes:
    institution:  MERCATOR OCEAN
    references:   http://www.mercator-ocean.fr
    source:       MERCATOR GLORYS12V1
    title:        daily mean fields from Global Ocean Physics Analysis and Fo...
    Conventions:  CF-1.4
    comment:      CMEMS product
    history:      2023/06/01 16:20:05 MERCATOR OCEAN Netcdf creation

#### Calculate the W

In [8]:
ds_i = prepare_ocean_dataset(ds)

In [9]:
print(ds_i)

<xarray.Dataset> Size: 52GB
Dimensions:      (time: 30, k: 50, j: 1201, i: 1201)
Coordinates: (12/17)
  * time         (time) datetime64[ns] 240B 1996-09-01 1996-09-02 ... 1996-09-30
  * k            (k) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * j            (j) int64 10kB 0 1 2 3 4 5 6 ... 1195 1196 1197 1198 1199 1200
  * i            (i) int64 10kB 0 1 2 3 4 5 6 ... 1195 1196 1197 1198 1199 1200
    depth_t      (k) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
    latitude_f   (j) float32 5kB -50.0 -49.92 -49.83 -49.75 ... 49.83 49.92 50.0
    ...           ...
    longitude_v  (i) float32 5kB -99.96 -99.88 -99.79 ... -0.04167 0.04167
    latitude_t   (j) float32 5kB -49.96 -49.88 -49.79 ... 49.88 49.96 50.04
    longitude_t  (i) float32 5kB -99.96 -99.88 -99.79 ... -0.04167 0.04167
    dz_t         (k) float32 200B 0.988 1.107 1.102 1.246 ... 435.3 447.7 458.6
    dx_t         (j) float64 10kB 5.961e+03 5.972e+03 ... 5.961e+03 5.951e+03
    dy_t     

In [10]:
import os
import dask
from tqdm.dask import TqdmCallback  # pip install tqdm

output_path = '/work/bk1450/b383184/Amazon/Atlantic/data/reanalysis'
os.makedirs(output_path, exist_ok=True)

var_to_file = {
    'uf': f'U_{start_date[:7]}.nc',
    'vf': f'V_{start_date[:7]}.nc',
    'w_c': f'W_{start_date[:7]}.nc',
}

tasks = []
for vname, fname in var_to_file.items():
    fullpath = os.path.join(output_path, fname)

    da = ds_i[vname].astype('float32')  # optional downcast
    enc = {
        vname: {
            'zlib': True, 'complevel': 4,
            'chunksizes': (1, 50, 512, 512),
        }
    }
    tasks.append(
        da.to_dataset(name=vname).to_netcdf(
            fullpath, engine='h5netcdf', encoding=enc, compute=False
        )
    )

with TqdmCallback(desc="Writing NetCDF files"):
    dask.compute(*tasks)

Writing NetCDF files:   0%|                                                  | 0/4636 [00:00<?, ?it/s]

Writing NetCDF files:   1%|▎                                        | 29/4636 [00:10<29:03,  2.64it/s]

Writing NetCDF files:   1%|▎                                        | 39/4636 [00:11<19:41,  3.89it/s]

Writing NetCDF files:   1%|▍                                        | 54/4636 [00:11<12:18,  6.20it/s]

Writing NetCDF files:   1%|▌                                        | 69/4636 [00:11<08:12,  9.27it/s]

...

Writing NetCDF files: 100%|██████████████████████████████████████▉| 4631/4636 [15:15<00:29,  5.80s/it]

Writing NetCDF files: 100%|██████████████████████████████████████▉| 4632/4636 [15:18<00:20,  5.23s/it]

Writing NetCDF files: 100%|██████████████████████████████████████▉| 4633/4636 [15:26<00:17,  5.95s/it]

Writing NetCDF files: 100%|██████████████████████████████████████▉| 4634/4636 [15:34<00:13,  6.58s/it]

Writing NetCDF files: 100%|███████████████████████████████████████| 4636/4636 [15:34<00:00,  4.96it/s]